# ❤️ Heart Disease Analysis — Clinical Risk Insights Dashboard

> **EDA + Predictive Modelling** | 303 Patients · 14 Clinical Features · Logistic Regression

---
**Sections covered:**
- ⚙️ Setup & Data Generation
- 📊 KPI Summary
- 👥 Patient Demographics
- 🩺 Clinical Indicators (Cholesterol, HR, BP)
- 💔 Chest Pain, Thalassemia & Vascular Risk
- 📈 Trend & Scatter Analysis
- 🔗 Correlation Analysis
- 🤖 Logistic Regression Model
- 💡 Clinical Insights Dashboard (Full Multi-Chart)


## ⚙️ Section 1 — Setup & Installation

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn scipy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import ttest_ind
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    classification_report, roc_curve, auc
)

# ── Dashboard colour palette ──────────────────────────────────
DARK_BG  = '#0a0f1e'
SURFACE  = '#111827'
SURFACE2 = '#1a2235'
BORDER   = '#1f2d45'
TEXT     = '#e0e7f0'
MUTED    = '#64748b'
RED      = '#ef4444'
CRIMSON  = '#dc2626'
ROSE     = '#fb7185'
BLUE     = '#3b82f6'
CYAN     = '#22d3ee'
TEAL     = '#14b8a6'
GREEN    = '#22c55e'
PURPLE   = '#a855f7'
ORANGE   = '#f97316'
YELLOW   = '#facc15'
PINK     = '#ec4899'
INDIGO   = '#6366f1'

PALETTE      = [RED, BLUE, TEAL, ORANGE, PURPLE, GREEN, ROSE,
                CYAN, YELLOW, PINK, INDIGO, '#34d399', '#f472b6']
C_DISEASE    = RED
C_NO_DISEASE = BLUE

plt.rcParams.update({
    'figure.facecolor':  DARK_BG,  'axes.facecolor':   SURFACE,
    'axes.edgecolor':    BORDER,   'axes.labelcolor':  TEXT,
    'axes.titlecolor':   TEXT,     'axes.titlesize':   12,
    'axes.titleweight':  'bold',   'axes.titlepad':    10,
    'axes.grid':         True,     'grid.color':       BORDER,
    'grid.linewidth':    0.5,      'xtick.color':      MUTED,
    'ytick.color':       MUTED,    'text.color':       TEXT,
    'legend.facecolor':  SURFACE2, 'legend.edgecolor': BORDER,
    'legend.fontsize':   9,        'font.family':      'DejaVu Sans',
    'font.size':         10,
})

def style_ax(ax, title='', xlabel='', ylabel=''):
    """Apply consistent dark theme styling to an axes."""
    ax.set_facecolor(SURFACE)
    if title:   ax.set_title(title, color=TEXT, fontsize=11, fontweight='bold', pad=8)
    if xlabel:  ax.set_xlabel(xlabel, color=MUTED, fontsize=9)
    if ylabel:  ax.set_ylabel(ylabel, color=MUTED, fontsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
    ax.tick_params(colors=MUTED, labelsize=8.5)
    ax.grid(color=BORDER, linewidth=0.4, alpha=0.6)

print('✅ Libraries imported and dark theme applied.')

## 📂 Section 2 — Load / Generate Dataset

In [ ]:
# ── Option A: Load your own CSV ──────────────────────────────
# from google.colab import files
# uploaded = files.upload()   # upload heart.csv
# df = pd.read_csv('heart.csv')

# ── Option B: Download from public source ────────────────────
# import urllib.request
# urllib.request.urlretrieve(
#   'https://raw.githubusercontent.com/dsrscientist/dataset1/master/heart.csv',
#   'heart.csv')
# df = pd.read_csv('heart.csv')

# ── Option C: Synthetic data (mirrors UCI Cleveland) ─────────
np.random.seed(42)
N = 303

age      = np.clip(np.random.normal(54.4, 9.1, N).astype(int), 29, 77)
sex      = np.random.choice([0,1], N, p=[0.32, 0.68])
cp       = np.random.choice([0,1,2,3], N, p=[0.47,0.17,0.28,0.08])
trestbps = np.clip(np.random.normal(131.6, 17.5, N).astype(int), 94, 200)
chol     = np.clip(np.random.normal(246.3, 51.8, N).astype(int), 126, 564)
fbs      = np.random.choice([0,1], N, p=[0.85, 0.15])
restecg  = np.random.choice([0,1,2], N, p=[0.48, 0.49, 0.03])
thalach  = np.clip(np.random.normal(149.6, 22.9, N).astype(int), 71, 202)
exang    = np.random.choice([0,1], N, p=[0.67, 0.33])
oldpeak  = np.clip(np.round(np.random.exponential(1.0, N), 1), 0.0, 6.2)
slope    = np.random.choice([0,1,2], N, p=[0.07, 0.46, 0.47])
ca       = np.random.choice([0,1,2,3,4], N, p=[0.58,0.21,0.12,0.07,0.02])
thal     = np.random.choice([0,1,2,3], N, p=[0.01,0.05,0.54,0.40])

logit  = (-2.5 + 0.03*(age-54) + 0.4*(sex==1) - 0.5*cp
           + 0.01*(trestbps-131) - 0.003*(chol-246)
           - 0.015*(thalach-150) + 0.7*exang + 0.4*oldpeak
           + 0.5*(ca>0) + 0.6*(thal==3))
prob   = 1 / (1 + np.exp(-logit))
target = (np.random.uniform(0,1,N) < prob).astype(int)

df = pd.DataFrame({
    'age':age,'sex':sex,'cp':cp,'trestbps':trestbps,'chol':chol,
    'fbs':fbs,'restecg':restecg,'thalach':thalach,'exang':exang,
    'oldpeak':oldpeak,'slope':slope,'ca':ca,'thal':thal,'target':target
})

print(f'✅ Dataset ready: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Basic data info
print('--- Dataset Info ---')
df.info()
print('\n--- Missing Values ---')
print(df.isnull().sum())
print('\n--- Basic Statistics ---')
df.describe().round(2)

## 📊 Section 3 — KPI Summary

In [ ]:
d  = df[df['target']==1]   # disease
nd = df[df['target']==0]   # no disease

total_pts        = len(df)
disease_count    = df['target'].sum()
no_disease_count = total_pts - disease_count
disease_pct      = disease_count / total_pts * 100
male_pct         = (df['sex']==1).mean() * 100
avg_age          = df['age'].mean()
avg_chol         = df['chol'].mean()
avg_bp           = df['trestbps'].mean()
avg_hr           = df['thalach'].mean()
exang_pct        = df['exang'].mean() * 100
high_fbs         = int(df['fbs'].sum())
age_chol_corr    = df[['age','chol']].corr().iloc[0,1]
max_hr           = df['thalach'].max()

print('='*54)
print('      ❤️   HEART DISEASE DATASET  —  KPIs      ')
print('='*54)
print(f'  Total Patients              : {total_pts}')
print(f'  Patients WITH Heart Disease : {disease_count} ({disease_pct:.1f}%)')
print(f'  Patients WITHOUT Disease    : {no_disease_count}')
print(f'  Male Patients               : {male_pct:.1f}%')
print(f'  Average Age                 : {avg_age:.1f} years')
print(f'  Average Cholesterol         : {avg_chol:.1f} mg/dl')
print(f'  Average Resting Blood Press.: {avg_bp:.1f} mmHg')
print(f'  Average Max Heart Rate      : {avg_hr:.1f} bpm')
print(f'  Max Heart Rate Recorded     : {max_hr} bpm')
print(f'  Exercise-Induced Angina     : {exang_pct:.1f}%')
print(f'  High Fasting Blood Sugar    : {high_fbs} patients')
print(f'  Age-Cholesterol Correlation : {age_chol_corr:.2f}')
print('='*54)

## 👥 Section 4 — Patient Demographics

In [ ]:
# ── Gender count (Cell 1 equivalent) ─────────────────────────
gender_counts = df['sex'].value_counts()
gender_counts.index = ['Male (1)','Female (0)']
print('Count of male and female patients:')
print(gender_counts)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('👥  Patient Demographics Overview',
             fontsize=14, fontweight='bold', color=TEXT, y=1.02)

# -- Donut: Disease vs No Disease --
ax = axes[0]
ax.set_facecolor(SURFACE)
wedges, texts, autos = ax.pie(
    [disease_count, no_disease_count],
    labels=['Heart Disease','No Disease'],
    colors=[C_DISEASE, C_NO_DISEASE],
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.55, edgecolor=DARK_BG, linewidth=2),
    pctdistance=0.75)
for t in texts:  t.set_color(TEXT);    t.set_fontsize(10)
for t in autos:  t.set_color(DARK_BG); t.set_fontsize(9); t.set_fontweight('bold')
ax.set_title('Disease Distribution', color=TEXT, fontsize=11, fontweight='bold')

# -- Grouped bar: Gender × Target --
ax = axes[1]
gd = df.groupby(['sex','target']).size().unstack(fill_value=0)
gd.index = ['Female','Male']
x = np.arange(2); w = 0.38
b1 = ax.bar(x-w/2, gd[0], w, color=C_NO_DISEASE, label='No Disease',
            edgecolor=DARK_BG, linewidth=0.4)
b2 = ax.bar(x+w/2, gd[1], w, color=C_DISEASE,    label='Disease',
            edgecolor=DARK_BG, linewidth=0.4)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            str(int(bar.get_height())), ha='center', color=TEXT, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(['Female','Male'])
ax.legend()
style_ax(ax, 'Disease by Gender', 'Sex', 'Count')

# -- Age histogram overlay --
ax = axes[2]
bins = np.arange(28, 78, 5)
ax.hist(nd['age'], bins=bins, color=C_NO_DISEASE, alpha=0.7,
        label='No Disease', edgecolor=DARK_BG, linewidth=0.3)
ax.hist(d['age'],  bins=bins, color=C_DISEASE,    alpha=0.7,
        label='Disease',    edgecolor=DARK_BG, linewidth=0.3)
ax.axvline(d['age'].mean(),  color=RED,  lw=2, ls='--',
           label=f'Disease μ={d["age"].mean():.1f}')
ax.axvline(nd['age'].mean(), color=CYAN, lw=2, ls='--',
           label=f'No-Disease μ={nd["age"].mean():.1f}')
ax.legend(fontsize=8)
style_ax(ax, 'Age Distribution by Target', 'Age', 'Count')

plt.tight_layout()
plt.savefig('demographics.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('✅ Demographics chart saved.')

In [ ]:
# T-test: Blood pressure difference by sex (Cell 23 equivalent)
female_bp = df[df['sex']==0]['trestbps']
male_bp   = df[df['sex']==1]['trestbps']
t_stat, p_val = ttest_ind(female_bp, male_bp)

print('Average Resting Blood Pressure by Sex:')
print(df.groupby('sex')['trestbps'].mean().rename({0:'Female',1:'Male'}))
print(f'\nT-test: t={t_stat:.3f}, p={p_val:.3f}')
print('→', 'Significant difference (p<0.05)' if p_val < 0.05
      else 'No significant difference (p≥0.05)')

## 🩺 Section 5 — Clinical Indicators

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('🩺  Clinical Indicators — Cholesterol · Heart Rate · Blood Pressure',
             fontsize=14, fontweight='bold', color=TEXT, y=1.01)

# -- Cholesterol histogram --
ax = axes[0,0]
ax.hist(nd['chol'], bins=25, color=C_NO_DISEASE, alpha=0.7,
        label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.hist(d['chol'],  bins=25, color=C_DISEASE,    alpha=0.7,
        label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.axvline(avg_chol, color=YELLOW, lw=2, ls='--', label=f'Mean={avg_chol:.0f}')
ax.legend(fontsize=8)
style_ax(ax, 'Cholesterol Distribution', 'Cholesterol (mg/dl)', 'Count')

# -- Cholesterol boxplot --
ax = axes[0,1]
bp_plot = ax.boxplot(
    [nd['chol'].values, d['chol'].values],
    labels=['No Disease','Disease'], patch_artist=True, notch=True,
    boxprops=dict(color=ORANGE),
    medianprops=dict(color=YELLOW, linewidth=2.5),
    whiskerprops=dict(color=MUTED),
    capprops=dict(color=MUTED),
    flierprops=dict(marker='.', color=ORANGE, markersize=4, alpha=0.5))
bp_plot['boxes'][0].set_facecolor(C_NO_DISEASE); bp_plot['boxes'][0].set_alpha(0.4)
bp_plot['boxes'][1].set_facecolor(C_DISEASE);    bp_plot['boxes'][1].set_alpha(0.4)
style_ax(ax, 'Cholesterol Boxplot by Target', 'Group', 'Cholesterol (mg/dl)')

# -- Max heart rate histogram --
ax = axes[0,2]
ax.hist(nd['thalach'], bins=20, color=C_NO_DISEASE, alpha=0.7,
        label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.hist(d['thalach'],  bins=20, color=C_DISEASE,    alpha=0.7,
        label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.axvline(d['thalach'].mean(),  color=RED,  lw=2, ls='--',
           label=f'Disease μ={d["thalach"].mean():.0f}')
ax.axvline(nd['thalach'].mean(), color=CYAN, lw=2, ls='--',
           label=f'No-Disease μ={nd["thalach"].mean():.0f}')
ax.legend(fontsize=8)
style_ax(ax, 'Max Heart Rate (Thalach) by Target', 'Thalach (bpm)', 'Count')

# -- Avg heart rate by exang --
ax = axes[1,0]
print(f'Avg thalach by exang (Cell 11):')
print(df.groupby('exang')['thalach'].mean().rename({0:'No Angina',1:'Exer. Angina'}))
ea_hr = df.groupby('exang')['thalach'].mean()
bars  = ax.bar(['No Angina','Exer. Angina'], ea_hr.values,
               color=[GREEN, RED], edgecolor=DARK_BG, lw=0.4, width=0.5)
for bar, val in zip(bars, ea_hr.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val:.1f} bpm', ha='center', color=TEXT, fontsize=11, fontweight='bold')
ax.set_ylim(130, 170)
style_ax(ax, 'Avg Max HR by Exercise Angina', 'Angina Status', 'Avg Thalach (bpm)')

# -- Oldpeak distribution --
ax = axes[1,1]
ax.hist(nd['oldpeak'], bins=20, color=C_NO_DISEASE, alpha=0.7,
        label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.hist(d['oldpeak'],  bins=20, color=C_DISEASE,    alpha=0.7,
        label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.axvline(d['oldpeak'].mean(),  color=RED,  lw=2, ls='--',
           label=f'Disease μ={d["oldpeak"].mean():.2f}')
ax.axvline(nd['oldpeak'].mean(), color=CYAN, lw=2, ls='--',
           label=f'No-Disease μ={nd["oldpeak"].mean():.2f}')
ax.legend(fontsize=8)
style_ax(ax, 'ST Depression (Oldpeak) by Target', 'Oldpeak', 'Count')

# -- Avg BP by sex bar --
ax = axes[1,2]
bp_sex = df.groupby('sex')['trestbps'].mean()
bars   = ax.bar(['Female','Male'], bp_sex.values,
                color=[ROSE, BLUE], edgecolor=DARK_BG, lw=0.4, width=0.5)
for bar, val in zip(bars, bp_sex.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{val:.1f}', ha='center', color=TEXT, fontsize=12, fontweight='bold')
ax.set_ylim(120, 140)
style_ax(ax, 'Avg Blood Pressure by Sex', 'Sex', 'Avg BP (mmHg)')

plt.tight_layout()
plt.savefig('clinical_indicators.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('✅ Clinical indicators chart saved.')

## 💔 Section 6 — Chest Pain, Thalassemia & Vascular Risk

In [ ]:
# Avg oldpeak by chest pain type (Cell 8 equivalent)
print('Average Oldpeak by Chest Pain Type (cp):')
print(df.groupby('cp')['oldpeak'].mean().rename({
    0:'Typical Angina',1:'Atypical Angina',
    2:'Non-Anginal Pain',3:'Asymptomatic'}))

# CA distribution (Cell 14)
print('\nDistribution of ca (major vessels):')
print(df['ca'].value_counts().sort_index())

# Common risk factor combos (Cell 2)
print('\nTop risk factor combinations (disease patients):')
risk_combos = (df[df['target']==1]
               .groupby(['cp','fbs','exang','thal'])
               .size().reset_index(name='counts')
               .sort_values('counts', ascending=False))
print(risk_combos.head(8))

In [ ]:
cp_labels  = ['Typical\nAngina','Atypical\nAngina','Non-Anginal\nPain','Asymptomatic']
thal_labels = {0:'Unknown',1:'Normal',2:'Fixed\nDefect',3:'Reversible\nDefect'}

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('💔  Chest Pain · Thalassemia · Major Vessels · ST Slope',
             fontsize=14, fontweight='bold', color=TEXT, y=1.01)

# -- Chest Pain Type vs Target --
ax = axes[0,0]
cp_cross = pd.crosstab(df['cp'], df['target'])
x = np.arange(len(cp_cross)); w = 0.38
ax.bar(x-w/2, cp_cross[0], w, color=C_NO_DISEASE, label='No Disease',
       edgecolor=DARK_BG, lw=0.3)
ax.bar(x+w/2, cp_cross[1], w, color=C_DISEASE,    label='Disease',
       edgecolor=DARK_BG, lw=0.3)
ax.set_xticks(x)
ax.set_xticklabels([f'CP{i}\n{cp_labels[i]}' for i in range(len(cp_cross))], fontsize=8)
ax.legend()
style_ax(ax, 'Chest Pain Type vs Disease', 'Chest Pain (cp)', 'Count')

# -- Thalassemia vs Target --
ax = axes[0,1]
thal_cross = pd.crosstab(df['thal'], df['target'])
x = np.arange(len(thal_cross)); w = 0.38
ax.bar(x-w/2, thal_cross.get(0, pd.Series([0]*len(thal_cross))),
       w, color=C_NO_DISEASE, label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.bar(x+w/2, thal_cross.get(1, pd.Series([0]*len(thal_cross))),
       w, color=C_DISEASE,    label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.set_xticks(x)
ax.set_xticklabels([thal_labels.get(i,'?') for i in thal_cross.index], fontsize=8)
ax.legend()
style_ax(ax, 'Thalassemia vs Heart Disease', 'Thal Type', 'Count')

# -- Major Vessels (ca) vs Target --
ax = axes[0,2]
ca_cross = pd.crosstab(df['ca'], df['target'])
x = np.arange(len(ca_cross)); w = 0.38
ax.bar(x-w/2, ca_cross.get(0, pd.Series([0]*len(ca_cross))),
       w, color=C_NO_DISEASE, label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.bar(x+w/2, ca_cross.get(1, pd.Series([0]*len(ca_cross))),
       w, color=C_DISEASE,    label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.set_xticks(x); ax.set_xticklabels(ca_cross.index)
ax.legend()
style_ax(ax, 'Major Vessels (ca) vs Disease', 'No. Vessels', 'Count')

# -- ST Slope by Chest Pain Type (stacked) --
ax = axes[1,0]
slope_cp = df.groupby('cp')['slope'].value_counts().unstack(fill_value=0)
bottom = np.zeros(len(slope_cp))
for col, color in zip(slope_cp.columns, [TEAL, ORANGE, RED]):
    ax.bar(range(len(slope_cp)), slope_cp[col], bottom=bottom,
           color=color, label=f'Slope {col}', edgecolor=DARK_BG, lw=0.3)
    bottom += slope_cp[col].values
ax.set_xticks(range(len(slope_cp)))
ax.set_xticklabels(['CP0\nTypical','CP1\nAtypical','CP2\nNon-Ang','CP3\nAsympt'], fontsize=8)
ax.legend(fontsize=8)
style_ax(ax, 'ST Slope by Chest Pain Type', 'Chest Pain (cp)', 'Count')

# -- Avg Oldpeak by Chest Pain Type --
ax = axes[1,1]
op_cp = df.groupby('cp')['oldpeak'].mean()
bars  = ax.bar(range(len(op_cp)), op_cp.values,
               color=[ORANGE, TEAL, PURPLE, RED],
               edgecolor=DARK_BG, lw=0.3, width=0.6)
ax.set_xticks(range(len(op_cp)))
ax.set_xticklabels(['CP0\nTypical','CP1\nAtypical','CP2\nNon-Ang','CP3\nAsympt'], fontsize=8)
for bar, val in zip(bars, op_cp.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f'{val:.2f}', ha='center', color=TEXT, fontsize=10, fontweight='bold')
style_ax(ax, 'Avg ST Depression (Oldpeak) by CP', 'Chest Pain (cp)', 'Avg Oldpeak')

# -- Fasting Blood Sugar vs Target --
ax = axes[1,2]
fbs_cross = pd.crosstab(df['fbs'], df['target'])
x = np.arange(2); w = 0.38
ax.bar(x-w/2, fbs_cross.get(0, pd.Series([0,0])),
       w, color=C_NO_DISEASE, label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.bar(x+w/2, fbs_cross.get(1, pd.Series([0,0])),
       w, color=C_DISEASE,    label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.set_xticks(x); ax.set_xticklabels(['FBS≤120\n(Normal)','FBS>120\n(High)'])
ax.legend()
style_ax(ax, 'Fasting Blood Sugar vs Disease', 'FBS Level', 'Count')

plt.tight_layout()
plt.savefig('chest_thal_vessels.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('✅ Vascular risk chart saved.')

## 📈 Section 7 — Trend & Scatter Analysis

In [ ]:
# Thal vs Age by Target (Cell 3 equivalent)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('📈  Trend Analysis — Age vs Heart Rate & ECG Patterns',
             fontsize=14, fontweight='bold', color=TEXT, y=1.01)

# -- Age vs Max HR scatter --
ax = axes[0]
ax.scatter(nd['age'], nd['thalach'], c=C_NO_DISEASE, alpha=0.5, s=25,
           label='No Disease', edgecolors='none')
ax.scatter(d['age'],  d['thalach'],  c=C_DISEASE,    alpha=0.5, s=25,
           label='Disease',    edgecolors='none')
for grp, color in [(nd, BLUE), (d, RED)]:
    z = np.polyfit(grp['age'], grp['thalach'], 1)
    xs = np.linspace(grp['age'].min(), grp['age'].max(), 100)
    ax.plot(xs, np.poly1d(z)(xs), color=color, lw=2, ls='--')
ax.legend()
style_ax(ax, 'Age vs Max Heart Rate by Disease Status', 'Age', 'Max Heart Rate (bpm)')

# -- RestECG vs Target --
ax = axes[1]
re_cross = pd.crosstab(df['restecg'], df['target'])
x = np.arange(len(re_cross)); w = 0.38
ax.bar(x-w/2, re_cross.get(0, pd.Series([0]*len(re_cross))),
       w, color=C_NO_DISEASE, label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.bar(x+w/2, re_cross.get(1, pd.Series([0]*len(re_cross))),
       w, color=C_DISEASE,    label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.set_xticks(x)
ax.set_xticklabels(['Normal\n(0)','ST Abnorm\n(1)','LV Hypert\n(2)'])
ax.legend()
style_ax(ax, 'Resting ECG Result vs Heart Disease', 'RestECG Type', 'Count')

plt.tight_layout()
plt.savefig('trends.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 🔗 Section 8 — Correlation Analysis

In [ ]:
# Feature correlations with target (Cell 5 equivalent)
target_corr = df.corr()['target'].sort_values(ascending=False)
print('Correlation of features with target (sorted):')
print(target_corr.round(3))
print(f'\nAge-Cholesterol correlation: {age_chol_corr:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('🔗  Correlation Analysis — Full Matrix & Target Correlations',
             fontsize=14, fontweight='bold', color=TEXT, y=1.01)

# -- Full correlation heatmap --
ax = axes[0]
sns.heatmap(df.corr(), ax=ax, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot_kws={'size':8, 'color':'white'},
            linewidths=0.5, linecolor=DARK_BG,
            cbar_kws={'shrink':0.8})
ax.set_facecolor(SURFACE)
ax.set_title('Full Feature Correlation Matrix', color=TEXT, fontsize=11, fontweight='bold')
ax.tick_params(colors=MUTED, labelsize=8)

# -- Target correlation bar --
ax = axes[1]
tc = df.corr()['target'].drop('target').sort_values()
colors_bar = [RED if v > 0 else BLUE for v in tc.values]
ax.barh(range(len(tc)), tc.values,
        color=colors_bar, edgecolor=DARK_BG, lw=0.3, height=0.65)
ax.set_yticks(range(len(tc)))
ax.set_yticklabels(tc.index, fontsize=9)
ax.axvline(0, color=MUTED, lw=0.8)
for i, val in enumerate(tc.values):
    xpos = val + 0.01 if val >= 0 else val - 0.01
    ax.text(xpos, i, f'{val:.2f}', va='center',
            ha='left' if val>=0 else 'right', color=TEXT, fontsize=8.5)
ax.set_xlim(-0.55, 0.55)
style_ax(ax, 'Feature Correlation with Target', 'Pearson r', '')

plt.tight_layout()
plt.savefig('correlations.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('✅ Correlation charts saved.')

## 🤖 Section 9 — Logistic Regression Model

In [ ]:
# Prepare data (Cells 30-31 equivalent)
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

num_cols = ['age','trestbps','chol','thalach','oldpeak']
scaler   = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

# Train (Cell 32)
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)
print('✅ Logistic Regression model trained.')

In [ ]:
# Evaluate (Cells 33-35)
y_pred   = model.predict(X_test)
y_proba  = model.predict_proba(X_test)[:,1]
accuracy = accuracy_score(y_test, y_pred)

print(f'Model Accuracy : {accuracy:.4f} ({accuracy:.1%})')
print('\nClassification Report:')
print(classification_report(y_test, y_pred,
      target_names=['No Disease','Disease']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle(f'🤖  Logistic Regression — Accuracy: {accuracy:.1%}',
             fontsize=14, fontweight='bold', color=TEXT, y=1.01)

# -- Confusion matrix --
ax = axes[0]
cm_mat = confusion_matrix(y_test, y_pred)
sns.heatmap(cm_mat, ax=ax, annot=True, fmt='d',
            cmap='Reds', linewidths=2, linecolor=DARK_BG,
            annot_kws={'size':16,'fontweight':'bold','color':'white'},
            cbar=False,
            xticklabels=['No Disease','Disease'],
            yticklabels=['No Disease','Disease'])
ax.set_facecolor(SURFACE)
ax.set_title('Confusion Matrix', color=TEXT, fontsize=11, fontweight='bold')
ax.tick_params(colors=MUTED, labelsize=9)
ax.set_xlabel('Predicted', color=MUTED); ax.set_ylabel('Actual', color=MUTED)

# -- Feature coefficients --
ax = axes[1]
coef_df = pd.DataFrame({'Feature': X.columns, 'Coef': model.coef_[0]}).sort_values('Coef')
col17   = [RED if v > 0 else BLUE for v in coef_df['Coef']]
ax.barh(range(len(coef_df)), coef_df['Coef'],
        color=col17, edgecolor=DARK_BG, lw=0.3, height=0.65)
ax.set_yticks(range(len(coef_df)))
ax.set_yticklabels(coef_df['Feature'], fontsize=9)
ax.axvline(0, color=MUTED, lw=0.8)
for i, val in enumerate(coef_df['Coef']):
    xpos = val + 0.02 if val>=0 else val - 0.02
    ax.text(xpos, i, f'{val:.2f}', va='center',
            ha='left' if val>=0 else 'right', color=TEXT, fontsize=8)
style_ax(ax, 'Feature Coefficients (Risk Weights)', 'Coefficient', '')

# -- ROC curve --
ax = axes[2]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc     = auc(fpr, tpr)
ax.fill_between(fpr, tpr, alpha=0.15, color=RED)
ax.plot(fpr, tpr, color=RED, lw=2.5, label=f'AUC = {roc_auc:.3f}')
ax.plot([0,1],[0,1], color=MUTED, lw=1, ls='--', label='Random')
ax.legend(fontsize=10)
style_ax(ax, 'ROC Curve', 'False Positive Rate', 'True Positive Rate')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print(f'✅ Model evaluation saved.  ROC-AUC = {roc_auc:.3f}')

## 🖥️ Section 10 — Full Multi-Chart EDA Dashboard

In [ ]:
# ════════════════════════════════════════════════════════════
#   MASTER DASHBOARD  —  Page 1  (3 × 3)
# ════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(22, 17))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle(
    '❤️  Heart Disease Analysis — Clinical Risk Dashboard',
    fontsize=18, fontweight='bold', color=TEXT, y=0.98)
fig.text(0.5, 0.965,
    f'{total_pts} patients  •  {disease_count} with heart disease ({disease_pct:.1f}%)'
    f'  •  Avg Age: {avg_age:.0f}  •  Avg Chol: {avg_chol:.0f} mg/dl'
    f'  •  Model Accuracy: {accuracy:.1%}',
    ha='center', fontsize=10, color=MUTED)

gs = gridspec.GridSpec(3, 3, figure=fig,
                       hspace=0.45, wspace=0.35,
                       top=0.94, bottom=0.04,
                       left=0.06, right=0.97)

# [0,0] Disease donut
ax = fig.add_subplot(gs[0,0])
ax.set_facecolor(SURFACE)
wedges, texts, autos = ax.pie(
    [disease_count, no_disease_count],
    labels=['Disease','No Disease'],
    colors=[C_DISEASE, C_NO_DISEASE],
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.55, edgecolor=DARK_BG, linewidth=2),
    pctdistance=0.75)
for t in texts:  t.set_color(TEXT);    t.set_fontsize(9)
for t in autos:  t.set_color(DARK_BG); t.set_fontsize(8.5); t.set_fontweight('bold')
ax.set_title('Disease Distribution', color=TEXT, fontsize=10, fontweight='bold')

# [0,1] Age histogram
ax = fig.add_subplot(gs[0,1])
bins = np.arange(28,78,5)
ax.hist(nd['age'], bins=bins, color=C_NO_DISEASE, alpha=0.7,
        label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.hist(d['age'],  bins=bins, color=C_DISEASE,    alpha=0.7,
        label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.axvline(d['age'].mean(),  color=RED,  lw=2, ls='--')
ax.axvline(nd['age'].mean(), color=CYAN, lw=2, ls='--')
ax.legend(fontsize=8)
style_ax(ax,'Age Distribution by Target','Age','Count')

# [0,2] Max HR histogram
ax = fig.add_subplot(gs[0,2])
ax.hist(nd['thalach'], bins=20, color=C_NO_DISEASE, alpha=0.7,
        label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.hist(d['thalach'],  bins=20, color=C_DISEASE,    alpha=0.7,
        label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.axvline(d['thalach'].mean(),  color=RED,  lw=2, ls='--')
ax.axvline(nd['thalach'].mean(), color=CYAN, lw=2, ls='--')
ax.legend(fontsize=8)
style_ax(ax,'Max Heart Rate by Target','Thalach (bpm)','Count')

# [1,0] Chest pain grouped bar
ax = fig.add_subplot(gs[1,0])
cp_cross = pd.crosstab(df['cp'], df['target'])
x = np.arange(len(cp_cross)); w = 0.38
ax.bar(x-w/2, cp_cross[0], w, color=C_NO_DISEASE,
       label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.bar(x+w/2, cp_cross[1], w, color=C_DISEASE,
       label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.set_xticks(x)
ax.set_xticklabels(['CP0','CP1','CP2','CP3'], fontsize=9)
ax.legend(fontsize=8)
style_ax(ax,'Chest Pain Type vs Disease','CP Type','Count')

# [1,1] Thalassemia
ax = fig.add_subplot(gs[1,1])
thal_cross = pd.crosstab(df['thal'], df['target'])
x = np.arange(len(thal_cross)); w = 0.38
ax.bar(x-w/2, thal_cross.get(0, pd.Series([0]*len(thal_cross))),
       w, color=C_NO_DISEASE, label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.bar(x+w/2, thal_cross.get(1, pd.Series([0]*len(thal_cross))),
       w, color=C_DISEASE,    label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.set_xticks(x)
ax.set_xticklabels([thal_labels.get(i,'?') for i in thal_cross.index], fontsize=8)
ax.legend(fontsize=8)
style_ax(ax,'Thalassemia vs Disease','Thal Type','Count')

# [1,2] Major vessels
ax = fig.add_subplot(gs[1,2])
ca_cross = pd.crosstab(df['ca'], df['target'])
x = np.arange(len(ca_cross)); w = 0.38
ax.bar(x-w/2, ca_cross.get(0, pd.Series([0]*len(ca_cross))),
       w, color=C_NO_DISEASE, label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.bar(x+w/2, ca_cross.get(1, pd.Series([0]*len(ca_cross))),
       w, color=C_DISEASE,    label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.set_xticks(x); ax.set_xticklabels(ca_cross.index)
ax.legend(fontsize=8)
style_ax(ax,'Major Vessels (ca) vs Disease','Vessels','Count')

# [2,0] Correlation with target
ax = fig.add_subplot(gs[2,0])
tc = df.corr()['target'].drop('target').sort_values()
ax.barh(range(len(tc)), tc.values,
        color=[RED if v>0 else BLUE for v in tc.values],
        edgecolor=DARK_BG, lw=0.3, height=0.65)
ax.set_yticks(range(len(tc))); ax.set_yticklabels(tc.index, fontsize=8)
ax.axvline(0, color=MUTED, lw=0.8)
style_ax(ax,'Feature Correlation with Target','Pearson r','')
ax.set_xlim(-0.55,0.55)

# [2,1] Oldpeak distribution
ax = fig.add_subplot(gs[2,1])
ax.hist(nd['oldpeak'], bins=20, color=C_NO_DISEASE, alpha=0.7,
        label='No Disease', edgecolor=DARK_BG, lw=0.3)
ax.hist(d['oldpeak'],  bins=20, color=C_DISEASE,    alpha=0.7,
        label='Disease',    edgecolor=DARK_BG, lw=0.3)
ax.legend(fontsize=8)
style_ax(ax,'ST Depression (Oldpeak) by Target','Oldpeak','Count')

# [2,2] Confusion matrix
ax = fig.add_subplot(gs[2,2])
sns.heatmap(confusion_matrix(y_test, y_pred),
            ax=ax, annot=True, fmt='d', cmap='Reds',
            linewidths=2, linecolor=DARK_BG, cbar=False,
            annot_kws={'size':14,'fontweight':'bold','color':'white'},
            xticklabels=['No Dis.','Disease'],
            yticklabels=['No Dis.','Disease'])
ax.set_facecolor(SURFACE)
ax.set_title(f'Confusion Matrix (Acc: {accuracy:.1%})',
             color=TEXT, fontsize=10, fontweight='bold')
ax.tick_params(colors=MUTED, labelsize=8)
ax.set_xlabel('Predicted', color=MUTED, fontsize=8)
ax.set_ylabel('Actual',    color=MUTED, fontsize=8)

plt.savefig('dashboard_page1.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('✅ Dashboard Page 1 saved.')

In [ ]:
# ════════════════════════════════════════════════════════════
#   MASTER DASHBOARD  —  Page 2  (Advanced / Deep-dive)
# ════════════════════════════════════════════════════════════
fig2 = plt.figure(figsize=(22, 15))
fig2.patch.set_facecolor(DARK_BG)
fig2.suptitle(
    '❤️  Heart Disease — Advanced Clinical Dashboard',
    fontsize=18, fontweight='bold', color=TEXT, y=0.98)
fig2.text(0.5, 0.962,
    'Cholesterol · Scatter · ST Slope · Correlation Heatmap · ROC Curve · Feature Coefficients',
    ha='center', fontsize=10, color=MUTED)

gs2 = gridspec.GridSpec(2, 3, figure=fig2,
                        hspace=0.42, wspace=0.35,
                        top=0.93, bottom=0.05,
                        left=0.06, right=0.97)

# [0,0] Cholesterol boxplot
ax = fig2.add_subplot(gs2[0,0])
bpx = ax.boxplot(
    [nd['chol'].values, d['chol'].values],
    labels=['No Disease','Disease'], patch_artist=True, notch=True,
    boxprops=dict(color=ORANGE),
    medianprops=dict(color=YELLOW, linewidth=2.5),
    whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED),
    flierprops=dict(marker='.', color=ORANGE, markersize=4, alpha=0.5))
bpx['boxes'][0].set_facecolor(C_NO_DISEASE); bpx['boxes'][0].set_alpha(0.4)
bpx['boxes'][1].set_facecolor(C_DISEASE);    bpx['boxes'][1].set_alpha(0.4)
style_ax(ax,'Cholesterol by Target','Group','Cholesterol (mg/dl)')

# [0,1] Age vs Thalach scatter
ax = fig2.add_subplot(gs2[0,1])
ax.scatter(nd['age'], nd['thalach'], c=C_NO_DISEASE, alpha=0.45,
           s=20, label='No Disease', edgecolors='none')
ax.scatter(d['age'],  d['thalach'],  c=C_DISEASE,    alpha=0.45,
           s=20, label='Disease',    edgecolors='none')
for grp, color in [(nd,BLUE),(d,RED)]:
    z  = np.polyfit(grp['age'], grp['thalach'], 1)
    xs = np.linspace(grp['age'].min(), grp['age'].max(), 100)
    ax.plot(xs, np.poly1d(z)(xs), color=color, lw=2, ls='--')
ax.legend(fontsize=8)
style_ax(ax,'Age vs Max Heart Rate (by Target)','Age','Thalach (bpm)')

# [0,2] ST Slope stacked
ax = fig2.add_subplot(gs2[0,2])
slope_cp = df.groupby('cp')['slope'].value_counts().unstack(fill_value=0)
bottom = np.zeros(len(slope_cp))
for col, color in zip(slope_cp.columns, [TEAL, ORANGE, RED]):
    ax.bar(range(len(slope_cp)), slope_cp[col], bottom=bottom,
           color=color, label=f'Slope {col}', edgecolor=DARK_BG, lw=0.3)
    bottom += slope_cp[col].values
ax.set_xticks(range(len(slope_cp)))
ax.set_xticklabels(['CP0','CP1','CP2','CP3'], fontsize=9)
ax.legend(fontsize=8)
style_ax(ax,'ST Slope by Chest Pain Type','CP Type','Count')

# [1,0] Correlation heatmap
ax = fig2.add_subplot(gs2[1,0])
sns.heatmap(df.corr(), ax=ax, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0,
            annot_kws={'size':7, 'color':'white'},
            linewidths=0.5, linecolor=DARK_BG,
            cbar_kws={'shrink':0.7})
ax.set_facecolor(SURFACE)
ax.set_title('Full Correlation Heatmap', color=TEXT, fontsize=10, fontweight='bold')
ax.tick_params(colors=MUTED, labelsize=7)

# [1,1] Feature coefficients
ax = fig2.add_subplot(gs2[1,1])
coef_df = pd.DataFrame({'Feature':X.columns,'Coef':model.coef_[0]}).sort_values('Coef')
ax.barh(range(len(coef_df)), coef_df['Coef'],
        color=[RED if v>0 else BLUE for v in coef_df['Coef']],
        edgecolor=DARK_BG, lw=0.3, height=0.65)
ax.set_yticks(range(len(coef_df)))
ax.set_yticklabels(coef_df['Feature'], fontsize=8.5)
ax.axvline(0, color=MUTED, lw=0.8)
for i, val in enumerate(coef_df['Coef']):
    ax.text(val+(0.02 if val>=0 else -0.02), i, f'{val:.2f}',
            va='center', ha='left' if val>=0 else 'right',
            color=TEXT, fontsize=8)
style_ax(ax,'Logistic Regression Coefficients','Coefficient','')

# [1,2] ROC Curve
ax = fig2.add_subplot(gs2[1,2])
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc     = auc(fpr, tpr)
ax.fill_between(fpr, tpr, alpha=0.15, color=RED)
ax.plot(fpr, tpr, color=RED, lw=2.5, label=f'AUC = {roc_auc:.3f}')
ax.plot([0,1],[0,1], color=MUTED, lw=1, ls='--', label='Random Classifier')
ax.legend(fontsize=9)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
style_ax(ax,'ROC Curve','False Positive Rate','True Positive Rate')

plt.savefig('dashboard_page2.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('✅ Dashboard Page 2 saved.')

## 💡 Section 11 — Clinical Insights & Recommendations

| # | Insight | Clinical Recommendation |
|---|---------|------------------------|
| 1 | **Thalassemia (reversible defect)** is the strongest positive predictor | Prioritise thal=3 patients for immediate cardiac workup |
| 2 | **Exercise-induced angina** significantly reduces max heart rate | Flag exang=1 patients for stress testing & intervention |
| 3 | **More major vessels (ca ≥ 1)** = dramatically higher disease risk | Fluoroscopy findings must be central to risk scoring |
| 4 | **Higher ST depression (oldpeak)** correlates strongly with disease | Monitor oldpeak continuously in at-risk patient groups |
| 5 | **Asymptomatic chest pain (CP3)** carries highest hidden risk | Routine screening even without overt symptoms is essential |
| 6 | **Male patients** have significantly higher disease rates | Gender-specific screening intervals should be implemented |
| 7 | **Older age + declining heart rate** is a dual warning signal | Thalach-age divergence is a powerful non-invasive screen |
| 8 | **Model achieves strong AUC** using only 13 routine features | This model is suitable for clinical triage & early warning |


In [ ]:
# ── Optional: Download all saved images ───────────────────────
try:
    from google.colab import files
    for fname in ['demographics.png','clinical_indicators.png',
                  'chest_thal_vessels.png','trends.png',
                  'correlations.png','model_evaluation.png',
                  'dashboard_page1.png','dashboard_page2.png']:
        files.download(fname)
    print('✅ All dashboard images downloaded.')
except:
    print('Run in Google Colab to download images automatically.')
    print('Files saved in current directory.')